In [1]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow

In [2]:
weekly_hvac_permits = pd.read_parquet("../data/weekly_hvac_permits.parquet")

In [3]:
weekly_hvac_permits.index.dtype

dtype('int64')

In [4]:
weekly_hvac_permits

,ds,y,unique_id
0,2010-01-10,265,la_hvac
1,2010-01-17,261,la_hvac
2,2010-01-24,196,la_hvac
3,2010-01-31,253,la_hvac
4,2010-02-07,259,la_hvac
...,...,...,...
867,2026-08-23,385,la_hvac
868,2026-08-30,372,la_hvac
869,2026-09-06,358,la_hvac
870,2026-09-13,299,la_hvac


In [5]:
mask = weekly_hvac_permits['ds'] < '2026-03-22'
print(mask)

0       True
1       True
2       True
3       True
4       True
       ...  
867    False
868    False
869    False
870    False
871    False
Name: ds, Length: 872, dtype: bool


In [6]:
weekly_hvac_permits.loc[mask]

,ds,y,unique_id
0,2010-01-10,265,la_hvac
1,2010-01-17,261,la_hvac
2,2010-01-24,196,la_hvac
3,2010-01-31,253,la_hvac
4,2010-02-07,259,la_hvac
...,...,...,...
840,2026-02-15,281,la_hvac
841,2026-02-22,281,la_hvac
842,2026-03-01,302,la_hvac
843,2026-03-08,282,la_hvac


In [7]:
used_set = weekly_hvac_permits.iloc[:-26]

In [8]:
used_set

,ds,y,unique_id
0,2010-01-10,265,la_hvac
1,2010-01-17,261,la_hvac
2,2010-01-24,196,la_hvac
3,2010-01-31,253,la_hvac
4,2010-02-07,259,la_hvac
...,...,...,...
841,2026-02-22,281,la_hvac
842,2026-03-01,302,la_hvac
843,2026-03-08,282,la_hvac
844,2026-03-15,270,la_hvac


In [9]:
origins = used_set['ds'].iloc[793:838:4]
print(origins)

793   2025-03-23
797   2025-04-20
801   2025-05-18
805   2025-06-15
809   2025-07-13
813   2025-08-10
817   2025-09-07
821   2025-10-05
825   2025-11-02
829   2025-11-30
833   2025-12-28
837   2026-01-25
Name: ds, dtype: datetime64[us]


In [10]:
mask = weekly_hvac_permits['ds'] <= origins.iloc[0]

In [14]:
train = weekly_hvac_permits.loc[mask]

In [15]:
train

,ds,y,unique_id
0,2010-01-10,265,la_hvac
1,2010-01-17,261,la_hvac
2,2010-01-24,196,la_hvac
3,2010-01-31,253,la_hvac
4,2010-02-07,259,la_hvac
...,...,...,...
789,2025-02-23,198,la_hvac
790,2025-03-02,259,la_hvac
791,2025-03-09,247,la_hvac
792,2025-03-16,258,la_hvac


In [52]:
akey = used_set.iloc[origins.index[0] + 1: origins.index[0] + 7]

In [53]:
akey

,ds,y,unique_id
794,2025-03-30,233,la_hvac
795,2025-04-06,238,la_hvac
796,2025-04-13,306,la_hvac
797,2025-04-20,242,la_hvac
798,2025-04-27,268,la_hvac
799,2025-05-04,249,la_hvac


In [54]:
akey_guesses = used_set['y'].iloc[origins.index[0] - (52 - 1) : origins.index[0] - (52 - 7)]

In [55]:
akey_guesses

742    229
743    308
744    328
745    301
746    305
747    245
Name: y, dtype: int64

In [56]:
akey['baseline'] = akey_guesses.tolist()

In [57]:
akey

,ds,y,unique_id,baseline
794,2025-03-30,233,la_hvac,229
795,2025-04-06,238,la_hvac,308
796,2025-04-13,306,la_hvac,328
797,2025-04-20,242,la_hvac,301
798,2025-04-27,268,la_hvac,305
799,2025-05-04,249,la_hvac,245


In [58]:
akey['fold'] = 1

In [66]:
akey['horizon'] = list(range(1, 7))

In [60]:
akey

,ds,y,unique_id,baseline,fold,horizon
794,2025-03-30,233,la_hvac,229,1,1
795,2025-04-06,238,la_hvac,308,1,2
796,2025-04-13,306,la_hvac,328,1,3
797,2025-04-20,242,la_hvac,301,1,4
798,2025-04-27,268,la_hvac,305,1,5
799,2025-05-04,249,la_hvac,245,1,6


In [67]:
folds = []
for i in range(12):
    mask = weekly_hvac_permits['ds'] <= origins.iloc[i]
    train = weekly_hvac_permits.loc[mask]
    akey = used_set.iloc[origins.index[i] + 1: origins.index[i] + 7]
    akey_guesses = used_set['y'].iloc[origins.index[i] - (52 - 1) : origins.index[i] - (52 - 7)]
    akey['baseline'] = akey_guesses.tolist()
    akey['fold'] = i + 1
    akey['horizon'] = list(range(1, 7))
    folds.append(akey)

print(len(folds))

12


In [68]:
folds[-1]

,ds,y,unique_id,baseline,fold,horizon
838,2026-02-01,248,la_hvac,248,12,1
839,2026-02-08,254,la_hvac,235,12,2
840,2026-02-15,281,la_hvac,261,12,3
841,2026-02-22,281,la_hvac,198,12,4
842,2026-03-01,302,la_hvac,259,12,5
843,2026-03-08,282,la_hvac,247,12,6
